In [7]:
import pandas as pd
tripfile =r"C:\Users\hyunj\Seoul_Strolling_Adventure\Crawling\전국.xlsx"
trip_Df = pd.read_excel(tripfile)

trip_Df.head(2)

,addr1,addr2,areacode,booktour,cat1,cat2,cat3,contentid,contenttypeid,createdtime,...,firstimage2,cpyrhtDivCd,mapx,mapy,mlevel,modifiedtime,sigungucode,tel,title,zipcode
0,충청남도 공주시 감영길 3,(반죽동),34.0,NaN,쇼핑,쇼핑,전문매장/상가,2750144,38,20210928012320,...,NaN,NaN,127.121672,36.452930,6.0,20241226163730,1.0,NaN,가가상점,32546
1,부산광역시 부산진구 중앙번영로 (6),NaN,6.0,NaN,음식,음식점,일식,2805408,39,20220125140006,...,NaN,NaN,129.059828,35.144807,6.0,20240104133528,7.0,NaN,가가와,47361


In [ ]:

import math, requests# 한국 경위도 범위
from config import API_KEY

KOREA_LON_MIN, KOREA_LON_MAX = 124.0, 132.0
KOREA_LAT_MIN, KOREA_LAT_MAX = 33.0, 39.0

def _in_korea(lon, lat):
    return (KOREA_LON_MIN <= lon <= KOREA_LON_MAX) and (KOREA_LAT_MIN <= lat <= KOREA_LAT_MAX)

def _maybe_swap(lon, lat):
    in_ok = _in_korea(lon, lat)
    swapped_ok = _in_korea(lat, lon)
    if (not in_ok) and swapped_ok:
        return lat, lon  # 뒤집어서 사용
    return lon, lat

def get_address_from_coords(x, y, api_key):
    # NaN 방어 및 형변환
    if pd.isna(x) or pd.isna(y):
        return None
    try:
        lon, lat = float(x), float(y)
    except Exception:
        return None

    # 한국 범위 기준 자동 스왑
    lon, lat = _maybe_swap(lon, lat)

    # 한국 범위를 완전히 벗어나면 시도하지 않음
    if not _in_korea(lon, lat):
        return None

    url = "https://dapi.kakao.com/v2/local/geo/coord2address.json"
    headers = {"Authorization": f"KakaoAK {api_key}"}
    params = {"x": lon, "y": lat, "input_coord": "WGS84"}  # 안전하게 명시
    try:
        res = requests.get(url, headers=headers, params=params, timeout=5)
        if res.status_code != 200:
            # 원인 파악을 위해 상태/메시지 한 번 출력
            print(f"[WARN] status={res.status_code} body={res.text[:120]}")
            return None
        docs = res.json().get("documents", [])
        if not docs:
            return None
        # address 없을 수도 있어 road_address로 폴백
        d0 = docs[0]
        addr = (d0.get("address") or {}).get("address_name")
        if not addr:
            addr = (d0.get("road_address") or {}).get("address_name")
        return addr
    except requests.RequestException as e:
        print(f"[ERR] request failed: {e}")
        return None

# === 메인 ===
before = trip_Df['addr1'].isna().sum()

# addr1이 NaN이고 mapx/mapy가 있는 행만 처리
na_idx = trip_Df.index[
    trip_Df['addr1'].isna() & trip_Df['mapx'].notna() & trip_Df['mapy'].notna()
]

filled = 0
for i in na_idx:
    x = trip_Df.at[i, 'mapx']
    y = trip_Df.at[i, 'mapy']
    addr = get_address_from_coords(x, y, API_KEY)
    if addr:
        trip_Df.at[i, 'addr1'] = addr
        filled += 1

after = trip_Df['addr1'].isna().sum()
print(f"📝 원래 {before}개 중 {filled}개 채웠고, {after}개 남음")

In [ ]:
trip_Df.to_csv("tourist_spots_with_addresses.csv", index=False, encoding="utf-8-sig")

In [ ]:
trip_Df = trip_Df[['addr1', 'cat1', 'cat2', 'cat3', 'mapx', 'mapy','title']]
trip_Df.head(2)

In [ ]:
subway_file =r"C:\Users\hyunj\Seoul_Strolling_Adventure\tourism_transport_mapper\대중교통위치\전체_도시철도역사정보_20250417.xlsx"
subway_Df = pd.read_excel(subway_file)
subway_Df.head(2)

In [ ]:
import pandas as pd
import requests
import time
from tqdm import tqdm
from config import API_KEY

# 1) Kakao API 키 직접 입력
SEARCH_URL  = "https://dapi.kakao.com/v2/local/search/keyword.json"
HEADERS     = {"Authorization": f"KakaoAK {API_KEY}"}

# 3) 결과 저장용 리스트
valid_rows = []

# 4) 반복: 각 행(row)에 대해 역명 조회 및 저장
for _, row in tqdm(subway_Df.iterrows(), total=len(subway_Df)):
    raw_name = row["역사명"]
    # '역' 접미사 자동 추가
    query = raw_name if raw_name.endswith("역") else raw_name + "역"
    
    # Kakao Map REST API 호출
    resp = requests.get(
        SEARCH_URL,
        headers=HEADERS,
        params={"query": query, "size": 1}
    )
    if resp.status_code != 200:
        continue
    
    docs = resp.json().get("documents", [])
    if not docs:
        continue
    
    doc = docs[0]
    # 지하철역만 필터 (SW8: 지하철역)
    if doc.get("category_group_code") != "SW8":
        continue
    
    # 유효 역이면 필요한 정보만 저장 (name에도 '역' 포함)
    valid_rows.append({
        "line":  row["노선명"],       # 노선명
        "name":  query,              # 역명('역' 접미사 포함)
        "code":  row["노선번호"],     # 노선번호
        "위도":  float(doc["y"]),    # API 반환 위도
        "경도":  float(doc["x"]),    # API 반환 경도
    })
    # 호출 제한 대응
    time.sleep(0.2)

# 5) DataFrame 생성 및 Excel 저장
df_valid = pd.DataFrame(valid_rows, columns=["line","name","code","위도","경도"])
out_file = r"C:\Users\hyunj\Seoul_Strolling_Adventure\tourism_transport_mapper\대중교통위치\filtered_stations.xlsx"
df_valid.to_excel(out_file, index=False)

print(f"✅ 유효한 역 {len(df_valid)}개가 'filtered_stations.xlsx'에 저장되었습니다.")

df_valid.head(2)

In [ ]:
# subway_file =r'C:\Users\hyunj\machin_prj\대중교통위치\전체역_업데이트.xlsx'
# subway_Df = pd.read_excel(subway_file)
# subway_Df.head(2)

In [8]:
subway_file = r"C:\Users\hyunj\Seoul_Strolling_Adventure\tourism_transport_mapper\대중교통위치\filtered_stations.xlsx"
df_valid = pd.read_excel(subway_file)

In [9]:
# 그룹별 순서(order) 컬럼 추가
df_valid['order'] = df_valid.groupby('name').cumcount() + 1

# 피벗 수행: station별 line을 line1, line2로 분리
df_pivot = df_valid.pivot(index='name', columns='order', values='line')
df_pivot.columns = [f'line{col}' for col in df_pivot.columns]
df_pivot = df_pivot.reset_index()
df_pivot


,name,line1,line2,line3,line4,line5
0,4.19민주묘지역,우이신설선,NaN,NaN,NaN,NaN
1,가능역,경원선,NaN,NaN,NaN,NaN
2,가락시장역,3호선,8호선,NaN,NaN,NaN
3,가산디지털단지역,경부선,7호선,NaN,NaN,NaN
4,가야대역,부산김해경전철,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
838,회현(남대문시장)역,4호선,NaN,NaN,NaN,NaN
839,효자역,의정부,NaN,NaN,NaN,NaN
840,효창공원앞역,경의중앙선,6호선,NaN,NaN,NaN
841,흑석(중앙대입구)역,서울 도시철도 9호선,NaN,NaN,NaN,NaN


In [10]:
bus_file = r"C:\Users\hyunj\Seoul_Strolling_Adventure\tourism_transport_mapper\대중교통위치\국토교통부_전국 버스정류장 위치정보_20241028.csv"

bus_Df = pd.read_csv(bus_file, encoding='cp949')
bus_DF =bus_Df[["정류장번호","정류장명","위도","경도","도시명"]]
bus_DF.head(2)

,정류장번호,정류장명,위도,경도,도시명
0,ADB354000001,길안정류장,36.458658,128.891228,경상북도 안동시
1,ADB354000002,고란.계명산휴양림입구,36.397708,128.924029,경상북도 안동시


In [11]:
subway_file = r"C:\Users\hyunj\Seoul_Strolling_Adventure\tourism_transport_mapper\대중교통위치\filtered_stations.xlsx"

subway_Df = pd.read_excel(subway_file)
subway_Df.head(2)

,line,name,code,위도,경도
0,경인선,간석역,I1101,37.464706,126.693519
1,경인선,개봉역,I1101,37.494642,126.858716


In [12]:
import numpy as np
from scipy.spatial import cKDTree
import pandas as pd

def latlon_to_cartesian(lat, lon):
    lat = np.radians(lat)
    lon = np.radians(lon)
    R = 6371  # 지구 반지름
    x = R * np.cos(lat) * np.cos(lon)
    y = R * np.cos(lat) * np.sin(lon)
    z = R * np.sin(lat)
    return np.stack((x, y, z), axis=-1)

def find_nearest_kdtree(trip_df, subway_df, bus_df):
    # NaN 제거: 좌표 결측치가 있는 지하철/버스/trip 데이터 제외
    subway_df = subway_df.dropna(subset=['위도', '경도']).reset_index(drop=True)
    bus_df = bus_df.dropna(subset=['위도', '경도']).reset_index(drop=True)
    trip_df_valid = trip_df.dropna(subset=['mapy', 'mapx']).reset_index(drop=False)

    # 좌표를 Cartesian으로 변환
    subway_cartesian = latlon_to_cartesian(subway_df['위도'].values, subway_df['경도'].values)
    bus_cartesian = latlon_to_cartesian(bus_df['위도'].values, bus_df['경도'].values)
    trip_cartesian = latlon_to_cartesian(trip_df_valid['mapy'].values, trip_df_valid['mapx'].values)

    # KDTree 구성
    subway_tree = cKDTree(subway_cartesian)
    bus_tree = cKDTree(bus_cartesian)

    # 거리 계산
    subway_dist, subway_idx = subway_tree.query(trip_cartesian, k=1)
    bus_dist, bus_idx = bus_tree.query(trip_cartesian, k=1)

    # 비교 후 결과 작성
    nearest_station = []
    for i in range(len(trip_df_valid)):
        if subway_dist[i] < bus_dist[i]:
            nearest = f"{subway_df.iloc[subway_idx[i]]['line']} {subway_df.iloc[subway_idx[i]]['name']}"
        else:
            nearest = bus_df.iloc[bus_idx[i]]['정류장명']
        nearest_station.append(nearest)

    # 결과를 원래 trip_Df에 병합
    trip_df_result = trip_df.copy()
    trip_df_result['nearest_station'] = np.nan
    trip_df_result.loc[trip_df_valid['index'], 'nearest_station'] = nearest_station

    return trip_df_result


In [13]:
print(trip_Df.head(2))


                   addr1  addr2  areacode  booktour cat1 cat2     cat3  \
0         충청남도 공주시 감영길 3  (반죽동)      34.0       NaN   쇼핑   쇼핑  전문매장/상가   
1  부산광역시 부산진구  중앙번영로 (6)    NaN       6.0       NaN   음식  음식점       일식   

   contentid  contenttypeid     createdtime  ... firstimage2 cpyrhtDivCd  \
0    2750144             38  20210928012320  ...         NaN         NaN   
1    2805408             39  20220125140006  ...         NaN         NaN   

         mapx       mapy  mlevel    modifiedtime  sigungucode  tel title  \
0  127.121672  36.452930     6.0  20241226163730          1.0  NaN  가가상점   
1  129.059828  35.144807     6.0  20240104133528          7.0  NaN   가가와   

  zipcode  
0   32546  
1   47361  

[2 rows x 21 columns]


In [14]:
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

R_EARTH_KM = 6371.0

# 위도/경도 → 3D 카테시안 (KDTree용)
def latlon_to_cartesian(lat, lon):
    lat = np.radians(lat)
    lon = np.radians(lon)
    x = R_EARTH_KM * np.cos(lat) * np.cos(lon)
    y = R_EARTH_KM * np.cos(lat) * np.sin(lon)
    z = R_EARTH_KM * np.sin(lat)
    return np.stack((x, y, z), axis=-1)

# KDTree에서 나온 "현 거리(km)" → 대권거리(km)
def chord_to_arc_km(chord_km):
    # 안정성: 범위를 살짝 클램프
    x = np.clip(chord_km / (2.0 * R_EARTH_KM), 0.0, 1.0)
    return 2.0 * R_EARTH_KM * np.arcsin(x)

# df_pivot → {역명: [노선들]} 사전
def make_station_lines(df_pivot: pd.DataFrame):
    line_cols = [c for c in df_pivot.columns if str(c).startswith("line")]
    if "name" not in df_pivot.columns:
        raise ValueError("df_pivot에 'name' 컬럼이 필요합니다.")
    return (
        df_pivot
        .set_index("name")[line_cols]
        .apply(lambda r: [v for v in r if pd.notna(v)], axis=1)
        .to_dict()
    )

def get_closest_subway_and_bus_batch(
    trip_df: pd.DataFrame,
    subway_df: pd.DataFrame,
    bus_df: pd.DataFrame,
    station_lines: dict,
    max_km: float = 2.0,   # ← 임계거리(km). 초과 시 NaN
):
    # 원본 보존
    trip_result = trip_df.copy()

    # 필수 좌표 결측 제거
    subway_df = subway_df.dropna(subset=["위도", "경도"]).reset_index(drop=True)
    bus_df    = bus_df.dropna(subset=["위도", "경도"]).reset_index(drop=True)
    trip_valid = trip_df.dropna(subset=["mapy", "mapx"]).reset_index(drop=False)  # 원래 인덱스 보존

    # 기본 컬럼 준비
    for col in ["closest_subway_station", "closest_subway_line", "closest_bus_station", "closest_bus_id"]:
        if col not in trip_result.columns:
            trip_result[col] = np.nan

    # 지하철/버스 데이터가 비어 있으면 바로 리턴
    if len(subway_df) == 0 and len(bus_df) == 0:
        return trip_result

    # 좌표 → 3D
    trip_xyz = latlon_to_cartesian(trip_valid["mapy"].values, trip_valid["mapx"].values)

    # ===== 지하철 최근접 =====
    if len(subway_df) > 0:
        subway_xyz = latlon_to_cartesian(subway_df["위도"].values, subway_df["경도"].values)
        subway_tree = cKDTree(subway_xyz)
        subway_chord_km, subway_idx = subway_tree.query(trip_xyz, k=1)
        subway_arc_km = chord_to_arc_km(subway_chord_km)

        # 최근접 역/노선
        closest_subway_station = subway_df.iloc[subway_idx]["name"].values
        # 거리 2km 초과 → NaN
        mask_far_sub = subway_arc_km > max_km
        closest_subway_station = np.where(mask_far_sub, np.nan, closest_subway_station)

        # 노선 리스트(2km 이내만)
        closest_subway_lines = [
            (station_lines.get(st, []) if (isinstance(st, str) and st == st) else np.nan)
            for st in closest_subway_station
        ]

        # 결과 주입
        trip_result.loc[trip_valid["index"], "closest_subway_station"] = closest_subway_station
        trip_result.loc[trip_valid["index"], "closest_subway_line"] = pd.Series(
            closest_subway_lines, index=trip_valid["index"]
        )
    else:
        # 지하철 데이터가 없으면 전부 NaN 유지
        pass

    # ===== 버스 최근접 =====
    if len(bus_df) > 0:
        bus_xyz = latlon_to_cartesian(bus_df["위도"].values, bus_df["경도"].values)
        bus_tree = cKDTree(bus_xyz)
        bus_chord_km, bus_idx = bus_tree.query(trip_xyz, k=1)
        bus_arc_km = chord_to_arc_km(bus_chord_km)

        # 이름/ID 매핑 (같은 인덱스 기반)
        closest_bus_station = bus_df.iloc[bus_idx]["정류장명"].values
        closest_bus_id      = bus_df.iloc[bus_idx]["정류장번호"].astype(str).values

        # 2km 초과 → NaN 처리
        mask_far_bus = bus_arc_km > max_km
        closest_bus_station = np.where(mask_far_bus, np.nan, closest_bus_station)
        closest_bus_id      = np.where(mask_far_bus, np.nan, closest_bus_id)

        # 결과 주입
        trip_result.loc[trip_valid["index"], "closest_bus_station"] = closest_bus_station
        trip_result.loc[trip_valid["index"], "closest_bus_id"] = closest_bus_id
    else:
        # 버스 데이터가 없으면 전부 NaN 유지
        pass

    return trip_result


In [15]:
# 1) df_pivot → 노선 사전
station_lines = make_station_lines(df_pivot)

# 2) 2km 임계 적용해서 매핑
trip_Df = get_closest_subway_and_bus_batch(
    trip_Df, subway_Df, bus_DF, station_lines, max_km=2.0
)

# 3) 확인
trip_Df[["title","closest_subway_station","closest_subway_line",
         "closest_bus_station","closest_bus_id"]].head(10)


C:\Users\hyunj\AppData\Local\Temp\ipykernel_20388\710288288.py:81: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan '범내골역' nan ... '강동역' nan nan]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  trip_result.loc[trip_valid["index"], "closest_subway_station"] = closest_subway_station
C:\Users\hyunj\AppData\Local\Temp\ipykernel_20388\710288288.py:82: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan list(['부산 도시철도 1호선']) nan ... list(['5호선']) nan nan]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  trip_result.loc[trip_valid["index"], "closest_subway_line"] = pd.Series(
C:\Users\hyunj\AppData\Local\Temp\ipykernel_20388\710288288.py:106: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future

,title,closest_subway_station,closest_subway_line,closest_bus_station,closest_bus_id
0,가가상점,NaN,NaN,사대부고,SJB286034019
1,가가와,범내골역,[부산 도시철도 1호선],중앙시장,BSB510100000
2,가가책방,NaN,NaN,사대부고,SJB286034019
3,가거도(소흑산도),NaN,NaN,NaN,NaN
4,가경 터미널시장,NaN,NaN,서부소방서.가경터미널시장,CJB271000059
5,가경목장,사리역,[수인선],감골길앞,GGB216000314
6,가경식당,NaN,NaN,부여여자고등학교,TSB294001193
7,가경재,NaN,NaN,하회마을,ADB354002046
8,가계해수욕장,NaN,NaN,가계승강장,TSB344000018
9,가고파 꼬부랑길 벽화마을,NaN,NaN,창원시립마산박물관,CWB379005472


In [16]:
trip_Df['closest_bus_station'] = trip_Df['closest_bus_station'].str.replace(r'\(.*\)$', '', regex=True)
trip_Df.head(2)

,addr1,addr2,areacode,booktour,cat1,cat2,cat3,contentid,contenttypeid,createdtime,...,mlevel,modifiedtime,sigungucode,tel,title,zipcode,closest_subway_station,closest_subway_line,closest_bus_station,closest_bus_id
0,충청남도 공주시 감영길 3,(반죽동),34.0,NaN,쇼핑,쇼핑,전문매장/상가,2750144,38,20210928012320,...,6.0,20241226163730,1.0,NaN,가가상점,32546,NaN,NaN,사대부고,SJB286034019
1,부산광역시 부산진구 중앙번영로 (6),NaN,6.0,NaN,음식,음식점,일식,2805408,39,20220125140006,...,6.0,20240104133528,7.0,NaN,가가와,47361,범내골역,[부산 도시철도 1호선],중앙시장,BSB510100000


In [ ]:
# station_choices = bus_DF['정류장명'].unique()
# station_choices

In [17]:
multi_line_df = trip_Df[
    trip_Df['closest_subway_line'].apply(lambda x: isinstance(x, list) and len(x) >= 2)
]
print(multi_line_df[['title', 'closest_subway_station', 'closest_subway_line']].head(5))


       title closest_subway_station         closest_subway_line
18  가까운약국 서면                    서면역  [부산 도시철도 1호선, 부산 도시철도 2호선]
25     가나안약국                    신사역                 [3호선, 신분당선]
26   가나안장어마을                    풍산역                [경의중앙선, 서해선]
30      가넷옴므               가산디지털단지역                  [경부선, 7호선]
54    가락관광호텔                  가락시장역                  [3호선, 8호선]


In [18]:
result_df = trip_Df

In [19]:
output_path = r"C:\Users\hyunj\Seoul_Strolling_Adventure\tourism_transport_mapper\대중교통위치\관광지_대중교통_매핑결과.xlsx"

# 엑셀로 저장
result_df.to_excel(output_path, index=False, engine='xlsxwriter')
result_df.head(2)
print(f"✅ 결과가 '{output_path}'에 저장되었습니다.")

c:\Users\hyunj\anaconda3\envs\env1\Lib\site-packages\xlsxwriter\worksheet.py:1267: UserWarning: Ignoring URL 'http://tong.visitkorea.or.kr/cms/resource/09/2799309_image3_1.JPG' since it exceeds Excel's limit of 65,530 URLs per worksheet.
  warn(
c:\Users\hyunj\anaconda3\envs\env1\Lib\site-packages\xlsxwriter\worksheet.py:1267: UserWarning: Ignoring URL 'http://tong.visitkorea.or.kr/cms/resource/73/2899273_image3_1.jpg' since it exceeds Excel's limit of 65,530 URLs per worksheet.
  warn(
c:\Users\hyunj\anaconda3\envs\env1\Lib\site-packages\xlsxwriter\worksheet.py:1267: UserWarning: Ignoring URL 'http://tong.visitkorea.or.kr/cms/resource/69/2787369_image2_1.jpg' since it exceeds Excel's limit of 65,530 URLs per worksheet.
  warn(
c:\Users\hyunj\anaconda3\envs\env1\Lib\site-packages\xlsxwriter\worksheet.py:1267: UserWarning: Ignoring URL 'http://tong.visitkorea.or.kr/cms/resource/43/3465043_image3_1.JPG' since it exceeds Excel's limit of 65,530 URLs per worksheet.
  warn(
c:\Users\hyunj\a

✅ 결과가 'C:\Users\hyunj\Seoul_Strolling_Adventure\tourism_transport_mapper\대중교통위치\관광지_대중교통_매핑결과.xlsx'에 저장되었습니다.


In [20]:
import pandas as pd
result_file =r"C:\Users\hyunj\Seoul_Strolling_Adventure\tourism_transport_mapper\대중교통위치\관광지_대중교통_매핑결과.xlsx"
result_df = pd.read_excel(result_file)

result_df.head(2)

,addr1,addr2,areacode,booktour,cat1,cat2,cat3,contentid,contenttypeid,createdtime,...,mlevel,modifiedtime,sigungucode,tel,title,zipcode,closest_subway_station,closest_subway_line,closest_bus_station,closest_bus_id
0,충청남도 공주시 감영길 3,(반죽동),34.0,NaN,쇼핑,쇼핑,전문매장/상가,2750144,38,20210928012320,...,6.0,20241226163730,1.0,NaN,가가상점,32546,NaN,NaN,사대부고,SJB286034019
1,부산광역시 부산진구 중앙번영로 (6),NaN,6.0,NaN,음식,음식점,일식,2805408,39,20220125140006,...,6.0,20240104133528,7.0,NaN,가가와,47361,범내골역,['부산 도시철도 1호선'],중앙시장,BSB510100000


In [21]:
from pathlib import Path

# 1) 그룹 키 결정 (contentid > title+addr1 > title > 첫 컬럼)
if {"title","addr1"}.issubset(result_df.columns):
    GROUP_KEYS = ["title","addr1"]
elif "title" in result_df.columns:
    GROUP_KEYS = ["title"]
else:
    GROUP_KEYS = [result_df.columns[0]]

# 2) 모든 컬럼 사용 + '정류장번호'만 리스트 집계
def _valid(x):
    if pd.isna(x): 
        return False
    xs = str(x).strip()
    return bool(xs) and xs.lower() != "nan"

def first_non_null(s: pd.Series):
    # 문자열/숫자 혼재해도 첫 유효값 반환
    for v in s:
        if _valid(v):
            return v
    return None

def unique_list_keep_order(s: pd.Series):
    seen, out = set(), []
    for v in s:
        if not _valid(v):
            continue
        vv = str(v).strip()
        if vv not in seen:
            seen.add(vv); out.append(vv)
    return out

def to_pylist_string(lst):
    return "[" + ", ".join(f"'{x}'" for x in lst) + "]"

agg = {}
for c in result_df.columns:
    if c in GROUP_KEYS:
        continue
    if c == "정류장번호":
        agg[c] = unique_list_keep_order     # <-- 오직 이 컬럼만 리스트
    else:
        agg[c] = "first"                    # <-- 나머지는 대표값 하나(중복 제거 효과)

g = result_df.groupby(GROUP_KEYS, as_index=False).agg(agg)

# 리스트를 문자열로 변환
if "정류장번호" in g.columns:
    g["정류장번호"] = g["정류장번호"].apply(unique_list_keep_order).apply(to_pylist_string)

# 3) 저장 (CSV가 빠름)
out_csv = Path("관광지_대중교통_매칭최종.xlsx")
g.to_excel(out_csv, index=False, engine='xlsxwriter')
g.head(2)


,title,addr1,addr2,areacode,booktour,cat1,cat2,cat3,contentid,contenttypeid,...,mapy,mlevel,modifiedtime,sigungucode,tel,zipcode,closest_subway_station,closest_subway_line,closest_bus_station,closest_bus_id
0,"(고성) 숨은 영화 촬영지가 궁금할 때, 고성 무비 로드",강원특별자치도 고성군 죽왕면 왕곡마을길 35,None,32.0,NaN,추천코스,가족코스,가족코스,2514959,25,...,38.340226,6.0,20230809091829,2.0,None,24746,None,None,None,None
1,(구)인천일본제58은행지점,인천광역시 중구 신포로23번길 69-1 중구음식업지부,None,2.0,0.0,인문(문화/예술/역사),역사관광지,유적지/사적지,945221,12,...,37.472867,6.0,20240603104004,10.0,None,22314,인천역,"['경인선', '수인선']",중구청입구,ICB161000894


In [22]:
# target = g 또는 df
target = g  # or df

# title 오타 대비
key = "title" if "title" in target.columns else "tilte"

# cat1~3 중 존재하는 것만 포함해서 복합 키 구성
cat_keys = [c for c in ["addr1","cat1", "cat2", "cat3"] if c in target.columns]
keys = [key] + cat_keys

# 같은 (title, cat1, cat2, cat3) 조합이 2개 이상인 행만 필터
subset = target[target.groupby(keys)[keys[0]].transform("size").ge(2)]

subset.head(3)  # 원하는 행 수로 조절

,title,addr1,addr2,areacode,booktour,cat1,cat2,cat3,contentid,contenttypeid,...,mapy,mlevel,modifiedtime,sigungucode,tel,zipcode,closest_subway_station,closest_subway_line,closest_bus_station,closest_bus_id
